In [2]:
import pandas as pd
from datasets import load_dataset
from pathlib import Path
import json
from typing import *

/Users/markjos/projects/tira_parser/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tira_db_dir = Path('~/projects/tira_parser/.backups/tira_db_backup_12_30_25')

df = pd.read_csv(tira_db_dir / 'sentences.csv')
df.head()

,id,elan_sentence,updated_sentence,translation,elan_gloss,assigned_to,checked_by_annotator,checked_by_pi
0,1452,án ɔ́ɟɔ́ kálvə́lɛ̀ðà nd̪ɔ̀bàgɛ̀,án ɔ́ɟɔ́ kálvə́lɛ̀ðà nd̪ɔ̀bàgɛ̀,Who will pull them (away) tomorrow?,NaN,mark,f,f
1,7,lídí lìcə̀lò,lídí lìcə̀lò,The pot is good,CLl.pot CLl-good,mark,f,f
2,1453,án ɔ́ɟɔ́ kálvə̀lɛ̀ðɔ́ nd̪ɔ̀bagɛ̀,án ɔ́ɟɔ́ kálvə̀lɛ̀ðɔ́ nd̪ɔ̀bagɛ̀,Who will pull them (towards) tomorrow?,NaN,jenny,f,f
3,12,lɛ̀dɛ̌,lɛ̀dɛ̌,coconut,NaN,mark,f,f
4,13,ɛ̀ðɛ̀,ɛ̀ðɛ̀,vegetable,NaN,james,f,f


In [4]:
lexemes_df = pd.read_csv(tira_db_dir / 'lexemes.csv')
parses_df = pd.read_csv(tira_db_dir / 'parses.csv')
sentence_words_df = pd.read_csv(tira_db_dir / 'sentence_words.csv')
wordforms_df = pd.read_csv(tira_db_dir / 'wordforms.csv')

sentence_words_df.head()

,id,position,checked_by_pi,sentence_id,wordform_id,chosen_parse_id
0,4,0,f,3,3,NaN
1,5,1,f,3,4,NaN
2,8,0,f,5,6,NaN
3,20,0,f,12,15,NaN
4,22,0,f,14,17,NaN


In [5]:
parses_df['analysis'] = parses_df['analysis'].apply(json.loads)

parses_df.head()

,id,updated_form,source,comment,wordform_id,lexeme_id,edits,analysis
0,1,ɛ̀bɛ̀,FST,NaN,1,1113,1.0,"{'case': 'nominative', 'root': 'ɛ̀bɛ̀', 'gloss..."
1,2,ɛ̀bɛ̀,FST,NaN,1,1113,1.0,"{'case': 'unmarked', 'root': 'ɛ̀bɛ̀', 'gloss':..."
2,3,ɛ̀ðɛ̀,FST,NaN,1,1136,1.0,"{'case': 'nominative', 'root': 'r̀ðɛ̀', 'gloss..."
3,4,ɛ̀lɛ̀,FST,NaN,1,1106,1.0,"{'case': 'accusative', 'root': 'ɛ̀là(2)', 'gl..."
4,5,ɛ̀lɛ̀,FST,NaN,1,1105,1.0,"{'case': 'accusative', 'root': 'ɛ̀là(1)', 'gl..."


In [6]:
def get_gloss_str_from_dict(
        analysis: Dict[str, str],
        verbose: bool = False,
) -> str:
    analysis = analysis.copy()
    gloss = analysis.pop('gloss')
    analysis.pop('root', None)
    analysis.pop('part_of_speech', None)
    analysis.pop('analyzed_form', None)
    keys = sorted(analysis.keys())
    if verbose:
        other_parts = [f'[{key}={analysis[key]}]' for key in keys]
        gloss_str = gloss + ''.join(other_parts)
    else:
        if 'class' in analysis:
            # Prepend 'CL' to class value
            analysis['class'] = f"CL{analysis['class']}"
        other_parts = [f'{analysis[key]}' for key in keys]
        gloss_str = '-'.join([gloss] + other_parts)

    return gloss_str
get_gloss_str_from_dict(parses_df['analysis'][0])

'bamboo.gate-nominative-singular'

In [7]:
ds = load_dataset('tira-parsing/tira-parsing')
ds = ds.rename_column('updated_txt', 'updated_text')
ds

DatasetDict({
    train: Dataset({
        features: ['orig_text', 'translation', 'checked_by_pi', 'checked_by_ra', 'reviewer', 'updated_text', 'updated_gloss'],
        num_rows: 7861
    })
})

In [13]:
def fetch_updated_sentence(example):
    sentence_mask = df['elan_sentence'] == example['orig_text']
    if not sentence_mask.any():
        return example
    updated_text = df.loc[sentence_mask, 'updated_sentence'].iloc[0]
    if updated_text == example['orig_text']:
        return example
    example['updated_text'] = updated_text
    sentence_id = df.loc[sentence_mask, 'id'].iloc[0]
    sentence_word_mask = sentence_words_df['sentence_id'] == sentence_id
    chosen_parse_subset = sentence_words_df.loc[sentence_word_mask].sort_values('position')
    chosen_parse_id = chosen_parse_subset['chosen_parse_id'].tolist()

    glossed_words = []
    for parse in chosen_parse_id:
        if pd.isna(parse):
            glossed_words.append('<NO_PARSE>')
            continue
        parse_mask = parses_df['id'] == parse
        analysis_dict = parses_df.loc[parse_mask, 'analysis'].iloc[0]
        gloss_str = get_gloss_str_from_dict(analysis_dict)
        glossed_words.append(gloss_str)
    example['updated_gloss'] = ' '.join(glossed_words)
    return example


In [14]:
ds = ds.map(fetch_updated_sentence)

Map: 100%|██████████| 7861/7861 [00:06<00:00, 1133.17 examples/s]


In [15]:
ds_pandas = ds['train'].to_pandas()
has_gloss = ds_pandas['updated_gloss'] != ''
ds_pandas[has_gloss].head()

,orig_text,translation,checked_by_pi,checked_by_ra,reviewer,updated_text,updated_gloss
217,lálvə́lɛ̂ðɔ́ nd̪ɔ̀bàgɛ̀,you (sg) will pull them tomorrow (towards),False,False,,lávə̀lɛ̀ðɔ́ nd̪ɔ̀bà,pull-CLl-ventive-imperfective tomorrow
219,ŋ̀gátɛ́və́lɛ̂ðɔ́ nd̪ɔ̀bàgɛ̌,he will pull you all tomorrow,False,False,,ŋ̀gátɛ́və́lɛ̂ðɔ́ nd̪ɔ̀bà,<NO_PARSE> tomorrow
220,ŋ̀gátɛ́və́lɛ̂ðɔ́r nd̪ɔ̀bàgɛ̀,he will pull us all tomorrow,False,False,,ŋ̀gátɛ́və́lɛ̂ðɔ́r nd̪ɔ̀bà,<NO_PARSE> tomorrow
256,álâvə̀lɛ̀ðɔ́ ðáŋâlà nd̪ɔ̀bàgɛ̀,we two will pull the sheep (towards) tomorrow,False,False,,álâvə̀lɛ̀ðɔ́ ðàŋàlà nd̪ɔ̀bàgɛ̀,<NO_PARSE> sheep-accusative-singular <NO_PARSE>
277,t̪âlvə̀lɛ̀ðɛ̀ ùnɛ̀rɛ̀,it (lion) was pulled (by someone) yesterday / ...,False,False,,t̪àvə́lɛ̀ðɛ̀ ùnɛ́ɾɛ́,pull-CLt̪-itive-perfective yesterday


In [16]:
ds_pandas[has_gloss].to_csv(tira_db_dir/'parsed_sentences.csv', index=False)